In [1]:
# ============================================
# Seq2Seq vs Seq2Seq+Attention (Char-level)
# - 목표: vanilla vs attention 성능 비교 체험
# ============================================

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate, Attention, Multiply, Lambda
from tensorflow.keras.preprocessing.sequence import pad_sequences
import os, random, string

# ======== 0. 전역 시드 고정 ===========
RANDOM_SEED = 0
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)  # 파이썬 해시 시드 고정
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ===== 1. 데이터 생성 =====
def generate_data(num_samples=8000, min_len=5, max_len=30, task='copy'):
    """
    무작위 소문자 문자열을 생성 (길이 min_len~max_len)
    task='copy'   : 출력 = 입력 그대로
    task='reverse': 출력 = 입력의 역순
    """
    alphabet = list(string.ascii_lowercase)
    data_in, data_out = [], []
    for _ in range(num_samples):
        length = random.randint(min_len, max_len)
        seq = ''.join(random.choice(alphabet) for _ in range(length))
        if task == 'copy':
            data_out.append(seq)
        elif task == 'reverse':
            data_out.append(seq[::-1])
        data_in.append(seq)
    return data_in, data_out

# ===== 2. 토큰화 =====
def build_tokenizer(char_list):
    # 특수 토큰 3개 + 알파벳 26개 = 29개 어휘
    vocab = ['<pad>', '<start>', '<end>'] + char_list
    char2idx = {c:i for i,c in enumerate(vocab)}
    idx2char = {i:c for c,i in char2idx.items()}
    return char2idx, idx2char

char_list = list(string.ascii_lowercase)
char2idx, idx2char = build_tokenizer(char_list)
vocab_size = len(char2idx)

# --- 인코딩 함수 분리: 인코더 소스 / 디코더 입력 / 디코더 타깃 ---
def encode_src(seq, maxlen):
    """인코더 입력: 특수 토큰 없이 원문만 패딩"""
    tokens = list(seq)
    return pad_sequences([[char2idx[c] for c in tokens]], maxlen=maxlen, padding='post')[0]

def encode_tgt_in(seq, maxlen):
    """디코더 입력: <start> + 원문 + <end>"""
    tokens = ['<start>'] + list(seq) + ['<end>']
    return pad_sequences([[char2idx[c] for c in tokens]], maxlen=maxlen, padding='post')[0]

def encode_tgt_out(seq, maxlen):
    """디코더 타깃: (첫 글자부터) 원문 + <end>  ※ <start>는 타깃에 넣지 않음"""
    tokens = list(seq) + ['<end>']
    return pad_sequences([[char2idx[c] for c in tokens]], maxlen=maxlen, padding='post')[0]

# ===== 3. 모델 학습 & 추론 =====
def train_and_test(task='copy'):
    print(f"\n\n=== {task.upper()} TASK ===")
    # 길이 5~30으로 학습
    train_in, train_out = generate_data(8000, min_len=5, max_len=30, task=task)

    # 학습 세트 기준 최대 길이 계산
    max_encoder_len = max(len(s) for s in train_in)            # <= 30
    max_decoder_len = max(len(s) for s in train_out) + 2       # <start>, <end> 포함

    # 정수 인코딩 (소스/디코더 분리)
    encoder_input_data = np.array([encode_src(s,      max_encoder_len) for s in train_in])
    decoder_input_data = np.array([encode_tgt_in(s,   max_decoder_len) for s in train_out])
    decoder_target_data = np.array([encode_tgt_out(s, max_decoder_len) for s in train_out])

    # ===== 패딩 마스킹(손실에서 <pad> 제외) =====
    pad_id = char2idx['<pad>']
    sample_weights = (decoder_target_data != pad_id).astype('float32')  # (batch, T)

    latent_dim = 128  # 임베딩/은닉 차원

    # ===== 기본 Seq2Seq =====
    # 인코더
    encoder_inputs = Input(shape=(None,), dtype='int32', name="enc_in")
    enc_emb_layer = Embedding(vocab_size, latent_dim, mask_zero=True, name="enc_emb")   # mask_zero=True
    enc_emb = enc_emb_layer(encoder_inputs)
    encoder_lstm = LSTM(latent_dim, return_state=True, name="enc_lstm")
    _, state_h, state_c = encoder_lstm(enc_emb)
    encoder_states = [state_h, state_c]

    # 디코더
    decoder_inputs = Input(shape=(None,), dtype='int32', name="dec_in")
    dec_emb_layer = Embedding(vocab_size, latent_dim, mask_zero=True, name="dec_emb")   # mask_zero=True
    dec_emb = dec_emb_layer(decoder_inputs)
    decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="dec_lstm")
    decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
    decoder_dense = Dense(vocab_size, activation='softmax', name="out_dense")
    decoder_outputs = decoder_dense(decoder_outputs)

    seq2seq_model = Model([encoder_inputs, decoder_inputs], decoder_outputs, name="seq2seq_basic")
    seq2seq_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print("Training: Basic Seq2Seq")
    seq2seq_model.fit(
        [encoder_input_data, decoder_input_data],
        np.expand_dims(decoder_target_data, -1),      # (batch, T, 1) for sparse targets
        sample_weight=sample_weights,                  # (batch, T)
        batch_size=128, epochs=15, verbose=0
    )

    # 기본 추론용 모델
    encoder_model_basic = Model(encoder_inputs, encoder_states)
    decoder_state_input_h = Input(shape=(latent_dim,))
    decoder_state_input_c = Input(shape=(latent_dim,))
    dec_states_inputs = [decoder_state_input_h, decoder_state_input_c]
    decoder_inputs_inf = Input(shape=(None,), dtype='int32')
    dec_emb2 = dec_emb_layer(decoder_inputs_inf)
    dec_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=dec_states_inputs)
    dec_outputs2 = decoder_dense(dec_outputs2)
    decoder_model_basic = Model([decoder_inputs_inf] + dec_states_inputs, [dec_outputs2, state_h2, state_c2])

    # ===== 어텐션 Seq2Seq =====
    # 인코더
    encoder_inputs_a = Input(shape=(None,), dtype='int32', name="enc_in_a")
    enc_emb_layer_a = Embedding(vocab_size, latent_dim, mask_zero=True, name="enc_emb_a")
    enc_emb_a = enc_emb_layer_a(encoder_inputs_a)
    encoder_lstm_a = LSTM(latent_dim, return_sequences=True, return_state=True, name="enc_lstm_a")
    encoder_outputs_a, state_h_a, state_c_a = encoder_lstm_a(enc_emb_a)
    encoder_states_a = [state_h_a, state_c_a]

    # (핵심) 패딩 구간을 0으로: enc_mask -> expand -> 곱
    enc_mask_exp = Lambda(lambda x: tf.cast(tf.expand_dims(tf.not_equal(x, 0), -1), tf.float32))(encoder_inputs_a)
    encoder_outputs_masked = Multiply()([encoder_outputs_a, enc_mask_exp])

    # 디코더 + 어텐션 (마스크 인자 없이, 값만 0으로)
    decoder_inputs_a = Input(shape=(None,), dtype='int32', name="dec_in_a")
    dec_emb_layer_a = Embedding(vocab_size, latent_dim, mask_zero=True, name="dec_emb_a")
    dec_emb_a = dec_emb_layer_a(decoder_inputs_a)
    decoder_lstm_a = LSTM(latent_dim, return_sequences=True, return_state=True, name="dec_lstm_a")
    decoder_outputs_a, _, _ = decoder_lstm_a(dec_emb_a, initial_state=encoder_states_a)

    attention_layer = Attention(name="attention")  # dot-product attention
    attn_output_a = attention_layer([decoder_outputs_a, encoder_outputs_masked])
    decoder_concat_a = Concatenate(axis=-1, name="concat")([decoder_outputs_a, attn_output_a])
    decoder_dense_a = Dense(vocab_size, activation='softmax', name="out_dense_a")
    decoder_outputs_a = decoder_dense_a(decoder_concat_a)

    attn_model = Model([encoder_inputs_a, decoder_inputs_a], decoder_outputs_a, name="seq2seq_attn")
    attn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print("Training: Attention Seq2Seq")
    attn_model.fit(
        [encoder_input_data, decoder_input_data],
        np.expand_dims(decoder_target_data, -1),
        sample_weight=sample_weights,
        batch_size=128, epochs=15, verbose=0
    )

    # 어텐션 추론용 모델
    encoder_model_attn = Model(encoder_inputs_a, [encoder_outputs_a, state_h_a, state_c_a])
    decoder_state_input_h_a = Input(shape=(latent_dim,))
    decoder_state_input_c_a = Input(shape=(latent_dim,))
    # 추론 시에도 패딩 0 마스킹을 곱해줌
    encoder_outputs_input_a = Input(shape=(max_encoder_len, latent_dim))
    encoder_mask_input = Input(shape=(max_encoder_len,), dtype='int32')  # 0/1 마스크 입력
    enc_mask_inf = Lambda(lambda x: tf.cast(tf.expand_dims(tf.not_equal(x, 0), -1), tf.float32))(encoder_mask_input)
    encoder_outputs_masked_inf = Multiply()([encoder_outputs_input_a, enc_mask_inf])

    decoder_inputs_inf_a = Input(shape=(None,), dtype='int32')
    dec_emb2_a = dec_emb_layer_a(decoder_inputs_inf_a)
    dec_outputs2_a, state_h2_a, state_c2_a = decoder_lstm_a(
        dec_emb2_a, initial_state=[decoder_state_input_h_a, decoder_state_input_c_a]
    )
    attn_out2_a = attention_layer([dec_outputs2_a, encoder_outputs_masked_inf])
    dec_concat2_a = Concatenate(axis=-1)([dec_outputs2_a, attn_out2_a])
    dec_outputs2_a = decoder_dense_a(dec_concat2_a)
    decoder_model_attn = Model(
        [decoder_inputs_inf_a, decoder_state_input_h_a, decoder_state_input_c_a,
         encoder_outputs_input_a, encoder_mask_input],
        [dec_outputs2_a, state_h2_a, state_c2_a]
    )

    # ===== 디코딩 함수 =====
    def decode_sequence_basic(input_seq):
        """학습 시 사용한 최대 디코더 길이(max_decoder_len)만큼 생성"""
        h, c = encoder_model_basic.predict(input_seq, verbose=0)
        target_seq = np.array([[char2idx['<start>']]], dtype=np.int32)
        decoded = ''
        for _ in range(max_decoder_len):
            output_tokens, h, c = decoder_model_basic.predict([target_seq, h, c], verbose=0)
            sampled_id = int(np.argmax(output_tokens[0, -1, :]))
            token = idx2char[sampled_id]
            if token == '<end>':
                break
            if token not in ('<pad>', '<start>'):
                decoded += token
            target_seq = np.array([[sampled_id]], dtype=np.int32)
        return decoded

    def decode_sequence_attn(input_seq):
        # 인코더 출력 + 마스크 준비
        enc_outs, h, c = encoder_model_attn.predict(input_seq, verbose=0)
        # 입력 마스크: 0이 아닌 토큰 위치를 1로
        inp_mask = (input_seq[0] != 0).astype(np.int32)  # shape (T_enc,)
        target_seq = np.array([[char2idx['<start>']]], dtype=np.int32)
        decoded = ''
        for _ in range(max_decoder_len):
            output_tokens, h, c = decoder_model_attn.predict(
                [target_seq, h, c, enc_outs, inp_mask[np.newaxis, :]], verbose=0
            )
            sampled_id = int(np.argmax(output_tokens[0, -1, :]))
            token = idx2char[sampled_id]
            if token == '<end>':
                break
            if token not in ('<pad>', '<start>'):
                decoded += token
            target_seq = np.array([[sampled_id]], dtype=np.int32)
        return decoded

    # ===== 테스트 =====
    # 길이 5
    sample5 = "abcde"
    test_input5 = np.array([encode_src(sample5, max_encoder_len)])
    print(f"\n입력(길이 5): {sample5}")
    print("기본 Seq2Seq 출력:", decode_sequence_basic(test_input5))
    print("어텐션 Seq2Seq 출력:", decode_sequence_attn(test_input5))

    # 길이 30 (고정: abcdefghijklmnopqrstuvwxyzabcd)
    sample30 = "abcdefghijklmnopqrstuvwxyzabcd"  # 26 + 4 = 30
    test_input30 = np.array([encode_src(sample30, max_encoder_len)])  # max_encoder_len(=30)로 패딩
    print(f"\n입력(길이 30): {sample30}")
    print("기본 Seq2Seq 출력:", decode_sequence_basic(test_input30))
    print("어텐션 Seq2Seq 출력:", decode_sequence_attn(test_input30))

# ===== 실행 =====
train_and_test('copy')
train_and_test('reverse')




=== COPY TASK ===
Training: Basic Seq2Seq
Training: Attention Seq2Seq

입력(길이 5): abcde
기본 Seq2Seq 출력: abced
어텐션 Seq2Seq 출력: abcde

입력(길이 30): abcdefghijklmnopqrstuvwxyzabcd
기본 Seq2Seq 출력: dbacvefgcxhuwoljevjeswfodtvsdv
어텐션 Seq2Seq 출력: abcdefghijklmnopqrstuvwxyzabcd


=== REVERSE TASK ===
Training: Basic Seq2Seq
Training: Attention Seq2Seq

입력(길이 5): abcde
기본 Seq2Seq 출력: edcba


어텐션 Seq2Seq 출력: edcba

입력(길이 30): abcdefghijklmnopqrstuvwxyzabcd
기본 Seq2Seq 출력: cdbyzasxvrlyxnofcpumnjiqmgvbaz
어텐션 Seq2Seq 출력: dcbazyxwvutsrqponmlkjihgfedcba
